# 03 · Our Own CNN, Built From Scratch 🧱  ·  *Role C*

**Your mission:** a compact **siamese / two-branch** CNN — the same little network looks at the
*pre* and *post* patch, and we compare the two feature vectors to guess the damage level.

Keep it small! A giant network will just memorize the few damaged buildings and overfit. 🤏
The architecture already lives in `src/models.py` (`SiameseScratchCNN`) — you train & tune it here.

In [ ]:
# Setup. We reuse the shared code so this notebook stays short and consistent with 04.
import sys, os; sys.path.append(os.path.abspath('..'))
from src import config, data_utils, models, train_utils
import torch
from torch.utils.data import DataLoader

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Training on:', device, '(GPU is much faster, but CPU works for a small subset)')

### Load the data (train & val = the *seen* disasters)
`DamageDataset` reads Role A's `manifest.csv` and hands us `(pre, post, label)` batches.
The held-out disaster is saved for notebook 04 — don't peek at it here! 🙈

In [ ]:
train_ds = data_utils.DamageDataset(split='train', augment=True)   # flips etc. for variety
val_ds   = data_utils.DamageDataset(split='val',   augment=False)
train_dl = DataLoader(train_ds, batch_size=config.BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(val_ds,   batch_size=config.BATCH_SIZE, shuffle=False)
print(len(train_ds), 'train patches /', len(val_ds), 'val patches')

### Meet the model 🤖
Two branches **share weights** (same eyes looking at both photos). We combine their features by
`[pre, post, |pre - post|]` — that absolute difference is literally the *change* signal.
Then a tiny MLP head turns that into 4 class scores.

In [ ]:
model = models.SiameseScratchCNN(num_classes=config.NUM_CLASSES).to(device)
print(model)
print('Size:', sum(p.numel() for p in model.parameters())/1e6, 'M parameters (nice and small)')

### Train it 🏋️
We use class weights from Role A so the model can't cheat by always saying 'no-damage'.
The shared `train_model` saves the best checkpoint to `checkpoints/scratch.pt` — notebook 04
loads that file for the big comparison.

In [ ]:
import numpy as np
from pathlib import Path
w_path = Path(config.DATA_DIR)/'processed'/'class_weights.npy'
class_weights = torch.tensor(np.load(w_path), dtype=torch.float32) if w_path.exists() else None
if class_weights is None:
    print('No class_weights.npy yet — training unweighted for now (ask Role A).')

history = train_utils.train_model(model, train_dl, val_dl, device,
                                  epochs=config.EPOCHS, class_weights=class_weights,
                                  ckpt_path='../checkpoints/scratch.pt')
train_utils.plot_history(history)   # loss + accuracy curves

### How did we do (on *seen* disasters)? 🔎
Accuracy alone lies when classes are imbalanced, so look at **per-class precision/recall/F1**
and the confusion matrix. Pay special attention to recall on `destroyed` — missing destroyed
buildings is the costly mistake in real life.

(The held-out-disaster test happens in notebook 04, where both models face off.)

In [ ]:
train_utils.evaluate(model, val_dl, device, class_names=config.DAMAGE_CLASSES)